# GlobalCLIP -- 03: Evaluate and Analyse Trained Models

Loads both trained models (Standard and QLayer), evaluates them on the
held-out test set, and generates detailed analysis plots.

**Analyses performed:**
1. Per-sequence Pearson r on log-fold-enrichment (Standard vs QLayer)
2. Protein ranking by mean mixing coefficient α
3. 223×223 α correlation matrix (which proteins co-activate?)
4. Prediction profile comparison (high-signal test examples)
5. QLayer phase polar plot (cooperative vs competitive clusters)
6. QLayer coupling matrix cos(φ_i − φ_j)


## Set-up

### Imports

In [ ]:
import pylbsr.notebooks
import pylbsr.misc

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
from dotmap import DotMap
from scipy.stats import pearsonr

from parnet_additional_utils import (
    load_parnet_model,
    ParnetModelName,
)
from globalclip_utils import (
    GlobalCLIPStandardModel,
    GlobalCLIPQLayerModel,
    GlobalCLIPDataset,
    compute_log_enrichment,
    collect_alpha,
    rank_proteins,
    alpha_correlation_matrix,
    evaluate_pearson,
    plot_alpha_heatmap,
    plot_top_proteins,
    plot_phase_polar,
    plot_coupling_heatmap,
    plot_pearson_distribution,
    load_run_config,
)


### Initialisation

In [ ]:
_notebook_name = "03_evaluate_and_analyze.py.ipynb"
_notebook_path = f"notebooks/globalclip/{_notebook_name}"

pylbsr.notebooks.enable_cell_timing_metadata(show=True)
logger = pylbsr.misc.init_logger(_notebook_name)
PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)
logger.info(f"Project directory: {PROJECT_DIR}")


### Parameters

In [ ]:
params_gpu_index          = 0

# Run IDs from training notebooks (must match params_run_id set there)
params_standard_run_id    = "globalclip.standard.v1"
params_qlayer_run_id      = "globalclip.qlayer.v1"

# Evaluation
params_batch_size         = 128
params_num_workers        = 4
params_n_profile_examples = 5    # how many example profiles to plot

# Analysis
params_top_n_proteins     = 30   # top proteins to show in bar plot
params_phase_label_thresh = 0.0  # label phases with mean_alpha >= this


### Filepaths and device

In [ ]:
pylbsr.misc.set_seed(42)

device = torch.device(f"cuda:{params_gpu_index}" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(params_gpu_index)
    logger.info(f"GPU: {torch.cuda.get_device_name(device)}")
else:
    logger.warning("No GPU — running on CPU.")

_fp_cfg = yaml.safe_load((PROJECT_DIR / "config" / "filepaths.yaml").read_text())
pretrained_model_name = ParnetModelName.PARNET_7M_0_0

def _res(p):
    p = Path(p)
    return p if p.is_absolute() else PROJECT_DIR / p

FILEPATHS = DotMap()
FILEPATHS.pretrained_model    = _res(_fp_cfg["models"][pretrained_model_name.value])
FILEPATHS.standard_run_dir    = PROJECT_DIR / _fp_cfg["results"]["standard_model"] / params_standard_run_id
FILEPATHS.qlayer_run_dir      = PROJECT_DIR / _fp_cfg["results"]["qlayer_model"]   / params_qlayer_run_id
FILEPATHS.analysis_dir        = PROJECT_DIR / _fp_cfg["results"]["analysis"]
FILEPATHS.rbp_names_file      = PROJECT_DIR / "results" / "globalclip" / "datasets" / "rbp_names.txt"
FILEPATHS.analysis_dir.mkdir(parents=True, exist_ok=True)

# Load run configs to find dataset paths
_std_cfg = load_run_config(FILEPATHS.standard_run_dir)
_qlayer_cfg = load_run_config(FILEPATHS.qlayer_run_dir)
FILEPATHS.dataset = Path(_std_cfg["dataset_path"])

for k, v in FILEPATHS.items():
    logger.info(f"{k:30s}: {v}")


## Load RBP names and test data

In [ ]:
rbp_names = FILEPATHS.rbp_names_file.read_text().strip().split("\n")
logger.info(f"Loaded {len(rbp_names)} RBP names.")

test_ds = GlobalCLIPDataset(FILEPATHS.dataset, split="test",
                             seq_len=600, total_key="globalCLIP")
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=params_batch_size, shuffle=False,
    num_workers=params_num_workers, pin_memory=torch.cuda.is_available()
)
logger.info(f"Test set: {len(test_ds)} samples")


## Load trained models

In [ ]:
logger.info("Loading pretrained PARNET backbone...")
parnet = load_parnet_model(
    pretrained_model_name,
    FILEPATHS.pretrained_model,
    dtype=torch.float32,
    device=device,
)
parnet.eval()

# ── Standard model ────────────────────────────────────────────────────────────
model_std = GlobalCLIPStandardModel(
    parnet_model=parnet,
    num_rbps=_std_cfg["params_num_rbps"],
    mix_hidden=_std_cfg["params_mix_hidden"],
).to(device)
model_std.load_state_dict(
    torch.load(FILEPATHS.standard_run_dir / "model.statedict.pt", map_location=device)
)
model_std.eval()
logger.info(f"Standard model loaded from {FILEPATHS.standard_run_dir}")

# ── QLayer model ──────────────────────────────────────────────────────────────
model_ql = GlobalCLIPQLayerModel(
    parnet_model=parnet,
    num_rbps=_qlayer_cfg["params_num_rbps"],
    mix_hidden=_qlayer_cfg["params_mix_hidden"],
    cnn_channels=_qlayer_cfg["params_cnn_channels"],
    cnn_kernel=_qlayer_cfg["params_cnn_kernel"],
    cnn_layers=_qlayer_cfg["params_cnn_layers"],
).to(device)
model_ql.load_state_dict(
    torch.load(FILEPATHS.qlayer_run_dir / "model.statedict.pt", map_location=device)
)
model_ql.eval()
logger.info(f"QLayer model loaded from {FILEPATHS.qlayer_run_dir}")


## 1 — Test-set Pearson r (Standard vs QLayer)

In [ ]:
logger.info("Evaluating Standard model on test set...")
mean_r_std, all_r_std = evaluate_pearson(model_std, test_loader, device)
logger.info(f"Standard   mean Pearson r = {mean_r_std:.4f}  (std={all_r_std.std():.4f})")

logger.info("Evaluating QLayer model on test set...")
mean_r_ql, all_r_ql = evaluate_pearson(model_ql, test_loader, device)
logger.info(f"QLayer     mean Pearson r = {mean_r_ql:.4f}  (std={all_r_ql.std():.4f})")

print(f"\n{'Model':<20s}  {'Mean Pearson r':>15s}  {'Median':>8s}  {'Std':>8s}")
print("-" * 58)
for name, all_r, mean_r in [("Standard", all_r_std, mean_r_std),
                              ("QLayer",   all_r_ql,  mean_r_ql)]:
    print(f"{name:<20s}  {mean_r:>15.4f}  {np.median(all_r):>8.4f}  {all_r.std():>8.4f}")


In [ ]:
fig = plot_pearson_distribution(all_r_std, all_r_ql)
fig.savefig(FILEPATHS.analysis_dir / "pearson_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

# Save numeric results
results = {
    "standard": {"mean_r": float(mean_r_std), "median_r": float(np.median(all_r_std)), "std_r": float(all_r_std.std())},
    "qlayer":   {"mean_r": float(mean_r_ql),  "median_r": float(np.median(all_r_ql)),  "std_r": float(all_r_ql.std())},
}
(FILEPATHS.analysis_dir / "test_results.json").write_text(json.dumps(results, indent=2))
logger.info(f"Results saved to {FILEPATHS.analysis_dir / 'test_results.json'}")


## 2 — Protein ranking by mixing coefficient

Which RBPs contribute most to the GlobalCLIP signal across all test sequences?

In [ ]:
logger.info("Collecting alpha matrices (Standard model)...")
alpha_std = collect_alpha(model_std, test_loader, device)
logger.info(f"alpha_std shape: {alpha_std.shape}")

log_scale_std = model_std.log_scale.exp().detach().cpu().numpy()
ranking_std = rank_proteins(alpha_std, rbp_names, log_scale=log_scale_std)

print("Top-20 proteins (Standard model):")
print(ranking_std.head(20).to_string(index=False))


In [ ]:
fig = plot_top_proteins(ranking_std, top_n=params_top_n_proteins)
fig.savefig(FILEPATHS.analysis_dir / "protein_ranking_standard.png", dpi=120, bbox_inches="tight")
plt.show()

ranking_std.to_csv(FILEPATHS.analysis_dir / "protein_ranking_standard.tsv", sep="\t", index=False)


## 3 — Alpha correlation matrix

Proteins with correlated mixing coefficients tend to co-activate on the same
sequences — they likely belong to the same RNP complex or bind related motifs.

In [ ]:
corr_std = alpha_correlation_matrix(alpha_std)

fig = plot_alpha_heatmap(corr_std, rbp_names=None)   # too many names for labels
fig.savefig(FILEPATHS.analysis_dir / "alpha_correlation_standard.png", dpi=150, bbox_inches="tight")
plt.show()

# Top-10 most correlated pairs (excluding self)
idx = np.triu_indices(223, k=1)
pairs = [(corr_std[i, j], rbp_names[i], rbp_names[j])
         for i, j in zip(idx[0], idx[1])]
pairs.sort(key=lambda x: abs(x[0]), reverse=True)
print("Top-10 most correlated protein pairs (Standard model):")
print(f"{'r':>7s}  {'Protein A':<25s}  {'Protein B'}")
print("-" * 65)
for r, a, b in pairs[:10]:
    print(f"{r:>7.3f}  {a:<25s}  {b}")


## 4 — Prediction profiles (high-signal examples)

Select the test sequences with the highest total GlobalCLIP signal and compare
the ground truth log-FE against both model predictions.

In [ ]:
# Collect all signal totals to find high-signal sequences
from globalclip_utils.datasets import GlobalCLIPDataset

signal_totals = []
with torch.no_grad():
    for batch in test_loader:
        signal_totals.extend(batch["signal"].squeeze(1).sum(-1).tolist())
signal_totals = np.array(signal_totals)
top_indices = signal_totals.argsort()[::-1][:params_n_profile_examples]
logger.info(f"Top-{params_n_profile_examples} signal totals: {signal_totals[top_indices]}")


In [ ]:
fig, axes = plt.subplots(params_n_profile_examples, 3,
                          figsize=(16, 3.5 * params_n_profile_examples),
                          sharex=True)

with torch.no_grad():
    for row, idx in enumerate(top_indices):
        sample   = test_ds[idx]
        seq      = sample["sequence"].unsqueeze(0).to(device)
        signal   = sample["signal"]
        control  = sample["control"]

        target   = compute_log_enrichment(signal, control).squeeze().numpy()
        signal_np = signal.squeeze().numpy()
        control_np = control.squeeze().numpy()

        pred_std, _ = model_std(seq)
        pred_ql,  _ = model_ql(seq)
        pred_std_np = pred_std.squeeze().cpu().numpy()
        pred_ql_np  = pred_ql.squeeze().cpu().numpy()

        r_std, _ = pearsonr(pred_std_np, target)
        r_ql,  _ = pearsonr(pred_ql_np,  target)

        pos = np.arange(600)
        kw  = dict(linewidth=0, alpha=0.8)

        axes[row, 0].fill_between(pos, signal_np, color="black",       **kw)
        axes[row, 0].fill_between(pos, control_np, color="tomato",     alpha=0.5, linewidth=0)
        axes[row, 0].set_ylabel(f"Sample {idx}\ncounts", fontsize=8)
        if row == 0:
            axes[row, 0].set_title("Ground truth (black=signal, red=control)")

        axes[row, 1].fill_between(pos, pred_std_np, color="steelblue", **kw)
        axes[row, 1].set_ylabel(f"r={r_std:.3f}", fontsize=8, color="steelblue")
        if row == 0:
            axes[row, 1].set_title("Standard model prediction")

        axes[row, 2].fill_between(pos, pred_ql_np,  color="darkorange", **kw)
        axes[row, 2].set_ylabel(f"r={r_ql:.3f}", fontsize=8, color="darkorange")
        if row == 0:
            axes[row, 2].set_title("QLayer model prediction")

plt.suptitle("Prediction profiles on high-signal test sequences", y=1.01)
plt.tight_layout()
plt.savefig(FILEPATHS.analysis_dir / "profile_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


## 5 — QLayer phase analysis

After training, each of the 223 proteins has a learned phase φ_i.
Proteins clustered at similar phases cooperate (constructive interference);
proteins at opposite phases compete or cancel (destructive interference —
the mechanism by which the QLayer suppresses background noise).

In [ ]:
logger.info("Collecting alpha for QLayer model...")
alpha_ql = collect_alpha(model_ql, test_loader, device)
ranking_ql = rank_proteins(alpha_ql, rbp_names)

phases = model_ql.qlayer.phase.detach().cpu().numpy()
mean_alpha_ql = alpha_ql.mean(0)

print(f"Phase range: [{phases.min():.3f}, {phases.max():.3f}] rad")
print(f"Phase std:   {phases.std():.3f} rad")

# Label the most important proteins
params_phase_label_thresh = float(np.percentile(mean_alpha_ql, 80))
print(f"Labeling proteins with mean_alpha >= {params_phase_label_thresh:.4f} (top 20%)")


In [ ]:
fig = plot_phase_polar(
    phases=phases,
    rbp_names=rbp_names,
    label_threshold=params_phase_label_thresh,
    alpha_values=mean_alpha_ql,
)
fig.savefig(FILEPATHS.analysis_dir / "qlayer_phase_polar.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Phase-sorted protein table
phase_df = pd.DataFrame({
    "protein":    rbp_names,
    "phase_rad":  phases,
    "mean_alpha": mean_alpha_ql,
}).sort_values("phase_rad").reset_index(drop=True)

print("Proteins sorted by phase (head = most constructive, tail = most destructive):")
print(phase_df[["protein","phase_rad","mean_alpha"]].head(10).to_string(index=False))
print("...")
print(phase_df[["protein","phase_rad","mean_alpha"]].tail(10).to_string(index=False))

phase_df.to_csv(FILEPATHS.analysis_dir / "qlayer_phases.tsv", sep="\t", index=False)


## 6 — QLayer coupling matrix

`cos(φ_i − φ_j)`: +1 = fully cooperative, −1 = fully competitive.

In [ ]:
coupling = model_ql.get_coupling_matrix().cpu().numpy()

fig = plot_coupling_heatmap(coupling)
fig.savefig(FILEPATHS.analysis_dir / "qlayer_coupling.png", dpi=150, bbox_inches="tight")
plt.show()

# Top cooperative pairs
idx = np.triu_indices(223, k=1)
coup_pairs = [(coupling[i, j], rbp_names[i], rbp_names[j])
              for i, j in zip(idx[0], idx[1])]
coup_pairs.sort(key=lambda x: x[0], reverse=True)

print("Top-10 cooperative protein pairs (cos(φ_i−φ_j) ≈ 1):")
for c, a, b in coup_pairs[:10]:
    print(f"  cos(Δφ)={c:+.3f}  {a}  ↔  {b}")

print("\nTop-10 competitive protein pairs (cos(φ_i−φ_j) ≈ −1):")
for c, a, b in coup_pairs[-10:][::-1]:
    print(f"  cos(Δφ)={c:+.3f}  {a}  ↔  {b}")


## 7 — QLayer alpha correlation (comparison with Standard)

In [ ]:
corr_ql = alpha_correlation_matrix(alpha_ql)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, corr, name in [
    (axes[0], corr_std, "Standard model"),
    (axes[1], corr_ql,  "QLayer model"),
]:
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f"Alpha correlation matrix\n{name}")

plt.tight_layout()
plt.savefig(FILEPATHS.analysis_dir / "alpha_correlation_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


## Summary table

In [ ]:
summary = pd.DataFrame([
    {
        "Model":           "Standard (MixCoeffHead + log_scale)",
        "Trainable params": sum(p.numel() for p in model_std.parameters() if p.requires_grad),
        "Mean Pearson r":   f"{mean_r_std:.4f}",
        "Median Pearson r": f"{np.median(all_r_std):.4f}",
    },
    {
        "Model":           "QLayer (QLayer + dilated CNN)",
        "Trainable params": sum(p.numel() for p in model_ql.parameters() if p.requires_grad),
        "Mean Pearson r":   f"{mean_r_ql:.4f}",
        "Median Pearson r": f"{np.median(all_r_ql):.4f}",
    },
])

print("\n=== EVALUATION SUMMARY ===")
print(summary.to_string(index=False))
print("\nAll plots saved to:", FILEPATHS.analysis_dir)
